In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import recall_score, precision_score, roc_auc_score, confusion_matrix
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

In [12]:
df = pd.read_csv("/content/credit_risk_dataset.csv")

In [3]:
df.head()

,person_age,person_income,person_home_ownership,person_emp_length,loan_intent,loan_grade,loan_amnt,loan_int_rate,loan_status,loan_percent_income,cb_person_default_on_file,cb_person_cred_hist_length
0,22,59000,RENT,123.0,PERSONAL,D,35000,16.02,1,0.59,Y,3
1,21,9600,OWN,5.0,EDUCATION,B,1000,11.14,0,0.10,N,2
2,25,9600,MORTGAGE,1.0,MEDICAL,C,5500,12.87,1,0.57,N,3
3,23,65500,RENT,4.0,MEDICAL,C,35000,15.23,1,0.53,N,2
4,24,54400,RENT,8.0,MEDICAL,C,35000,14.27,1,0.55,Y,4


In [4]:
df = df.reset_index(drop=True)

In [13]:
#df.describe()
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 32581 entries, 0 to 32580
Data columns (total 12 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   person_age                  32581 non-null  int64  
 1   person_income               32581 non-null  int64  
 2   person_home_ownership       32581 non-null  object 
 3   person_emp_length           31686 non-null  float64
 4   loan_intent                 32581 non-null  object 
 5   loan_grade                  32581 non-null  object 
 6   loan_amnt                   32581 non-null  int64  
 7   loan_int_rate               29465 non-null  float64
 8   loan_status                 32581 non-null  int64  
 9   loan_percent_income         32581 non-null  float64
 10  cb_person_default_on_file   32581 non-null  object 
 11  cb_person_cred_hist_length  32581 non-null  int64  
dtypes: float64(3), int64(5), object(4)
memory usage: 3.0+ MB


In [6]:
df_test = df.head(10)
df_test

,person_age,person_income,person_home_ownership,person_emp_length,loan_intent,loan_grade,loan_amnt,loan_int_rate,loan_status,loan_percent_income,cb_person_default_on_file,cb_person_cred_hist_length
0,22,59000,RENT,123.0,PERSONAL,D,35000,16.02,1,0.59,Y,3
1,21,9600,OWN,5.0,EDUCATION,B,1000,11.14,0,0.10,N,2
2,25,9600,MORTGAGE,1.0,MEDICAL,C,5500,12.87,1,0.57,N,3
3,23,65500,RENT,4.0,MEDICAL,C,35000,15.23,1,0.53,N,2
4,24,54400,RENT,8.0,MEDICAL,C,35000,14.27,1,0.55,Y,4
5,21,9900,OWN,2.0,VENTURE,A,2500,7.14,1,0.25,N,2
6,26,77100,RENT,8.0,EDUCATION,B,35000,12.42,1,0.45,N,3
7,24,78956,RENT,5.0,MEDICAL,B,35000,11.11,1,0.44,N,4
8,24,83000,RENT,8.0,PERSONAL,A,35000,8.90,1,0.42,N,2
9,21,10000,OWN,6.0,VENTURE,D,1600,14.74,1,0.16,N,3


In [52]:
df.shape, df.isna().sum().sort_values(ascending=False)

# null_summary = pd.DataFrame({
#     'null_count': df.isna().sum(),
#     'null_pct': (df.isna().sum()/df.shape[0]) * 100
# })
# print(null_summary,df.shape[0])
# df[df['loan_int_rate'].isna()]
pd.DataFrame(((df.isna().sum().sort_values(ascending=False)/df.shape[0]))*100 > 5).reset_index()[0].value_counts(dropna=False)

,count
0,
False,11
True,1


In [7]:
df = df.head(10)
# df_test
# df_test.groupby('person_age')['person_income'].count()
# df_test = df_test.groupby('person_age')['loan_status'].agg(["count", "sum"]).reset_index().rename(columns={'sum': 'bads', 'count': 'total'})
# df_test['goods'] = df_test['total'] - df_test['bads']
# total_goods = df_test['goods'].sum()
# total_bads = df_test['bads'].sum()
# df_test['pct_good'] = (df_test['goods'] / total_goods)
# df_test['pct_bad']=(df_test['bads'] / total_bads)
# df_test['woe'] = np.log((df_test['pct_good'] + 0.5) / (df_test['pct_bad'] + 0.5 ))
# df_test

In [ ]:
def calculate_woe(df, feature, target):
    woe_df = (
        df.groupby(feature)[target]
        .agg(["count", "sum"])
        .reset_index()
        .rename(columns={"sum": "bads", "count": "total"})
    )

    woe_df["goods"] = woe_df["total"] - woe_df["bads"]

    total_goods = woe_df["goods"].sum()
    total_bads = woe_df["bads"].sum()

    # woe_df["pct_goods"] = (woe_df["goods"] + 0.5) / (total_goods + 0.5)
    # woe_df["pct_bads"] = (woe_df["bads"] + 0.5) / (total_bads + 0.5)

    woe_df["woe"] = np.log(woe_df["pct_goods"] / woe_df["pct_bads"])

    return woe_df[[feature, "woe"]]

In [ ]:
age_bins = [0, 25, 35, 45, 55, np.inf]
df["person_age_bin"] = pd.cut(df["person_age"], bins=age_bins)
age_woe_table = calculate_woe(df, "person_age_bin", "loan_status")
print(age_woe_table)

  person_age_bin       woe
0    (0.0, 25.0]  0.111226
1   (25.0, 35.0]  0.747214
2   (35.0, 45.0]  1.845827
3   (45.0, 55.0]  1.845827
4    (55.0, inf]  1.845827


/tmp/ipykernel_6487/1778266787.py:3: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df.groupby(feature)[target]


### **Trying something different but of no use**

In [ ]:
df_x = df.drop(columns ='loan_status')
df_y = df['loan_status']

In [ ]:
df.info()
cat_cols = ['person_home_ownership', 'loan_intent', 'loan_grade', 'cb_person_default_on_file']
num_cols = df_x.columns.difference(cat_cols).tolist()
num_cols

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 32581 entries, 0 to 32580
Data columns (total 12 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   person_age                  32581 non-null  int64  
 1   person_income               32581 non-null  int64  
 2   person_home_ownership       32581 non-null  object 
 3   person_emp_length           31686 non-null  float64
 4   loan_intent                 32581 non-null  object 
 5   loan_grade                  32581 non-null  object 
 6   loan_amnt                   32581 non-null  int64  
 7   loan_int_rate               29465 non-null  float64
 8   loan_status                 32581 non-null  int64  
 9   loan_percent_income         32581 non-null  float64
 10  cb_person_default_on_file   32581 non-null  object 
 11  cb_person_cred_hist_length  32581 non-null  int64  
dtypes: float64(3), int64(5), object(4)
memory usage: 3.0+ MB


['cb_person_cred_hist_length',
 'loan_amnt',
 'loan_int_rate',
 'loan_percent_income',
 'person_age',
 'person_emp_length',
 'person_income']

In [ ]:
df_x = df_x.dropna()

In [ ]:
#x = df.drop(columns= ['loan_status','person_home_ownership','loan_intent','loan_grade','cb_person_default_on_file'])
#x = df.drop(columns = cat_cols)
preprocessor = ColumnTransformer([
    ('cat', OneHotEncoder(), cat_cols),
    ('num', 'passthrough', num_cols)
])

x = preprocessor.fit_transform(df_x)

In [ ]:
#x.info()
#y.info()
feature_names = preprocessor.get_feature_names_out()
feature_names

array(['cat__person_home_ownership_MORTGAGE',
       'cat__person_home_ownership_OTHER',
       'cat__person_home_ownership_OWN',
       'cat__person_home_ownership_RENT',
       'cat__loan_intent_DEBTCONSOLIDATION', 'cat__loan_intent_EDUCATION',
       'cat__loan_intent_HOMEIMPROVEMENT', 'cat__loan_intent_MEDICAL',
       'cat__loan_intent_PERSONAL', 'cat__loan_intent_VENTURE',
       'cat__loan_grade_A', 'cat__loan_grade_B', 'cat__loan_grade_C',
       'cat__loan_grade_D', 'cat__loan_grade_E', 'cat__loan_grade_F',
       'cat__loan_grade_G', 'cat__cb_person_default_on_file_N',
       'cat__cb_person_default_on_file_Y',
       'num__cb_person_cred_hist_length', 'num__loan_amnt',
       'num__loan_int_rate', 'num__loan_percent_income',
       'num__person_age', 'num__person_emp_length', 'num__person_income'],
      dtype=object)

In [ ]:
x_train, x_test, y_train, y_test = train_test_split(
    x, y, test_size=0.2, random_state=42
)

In [ ]:
print("x_train")
display(x_train)
print(f"--------------------------------")
print("x_test")
display(x_test)
print("--------------------------------")
print("y_train")
display(y_train)
print("--------------------------------")
print("y_test")
display(y_test)
print("--------------------------------")

x_train


,person_age,person_income,person_emp_length,loan_amnt,loan_int_rate,loan_percent_income,cb_person_cred_hist_length
14324,25,105000,2.0,9600,11.71,0.09,3
5698,23,150000,3.0,5000,10.65,0.03,2
22936,34,35000,1.0,8000,6.17,0.23,9
23963,31,68000,15.0,13600,7.88,0.20,9
5750,26,25300,2.0,5400,16.02,0.21,2
...,...,...,...,...,...,...,...
24543,28,74000,12.0,5450,12.67,0.07,6
6202,23,27000,6.0,6000,11.66,0.22,3
991,22,24000,1.0,6000,12.68,0.25,4
17959,30,225000,2.0,25000,12.42,0.11,10


--------------------------------
x_test


,person_age,person_income,person_emp_length,loan_amnt,loan_int_rate,loan_percent_income,cb_person_cred_hist_length
24777,27,75400,3.0,6000,7.74,0.08,9
18341,33,53004,0.0,20000,13.04,0.38,10
14584,26,110000,3.0,4000,6.92,0.04,2
23814,29,60000,11.0,6000,15.27,0.10,10
20906,28,66725,3.0,9000,16.29,0.11,9
...,...,...,...,...,...,...,...
27436,33,140000,8.0,8000,5.79,0.06,6
9169,26,16120,2.0,5000,11.58,0.31,3
17987,34,204000,0.0,16000,11.83,0.08,5
3532,22,37000,6.0,7500,10.95,0.20,2


--------------------------------
y_train


,loan_status
14324,0
5698,0
22936,0
23963,0
5750,1
...,...
24543,0
6202,0
991,0
17959,0


--------------------------------
y_test


,loan_status
24777,0
18341,1
14584,0
23814,1
20906,1
...,...
27436,0
9169,1
17987,0
3532,0


--------------------------------


In [ ]:
model = LogisticRegression(max_iter=1000)
model.fit(x_train, y_train)

LogisticRegression(max_iter=1000)

In [ ]:
y_pred = model.predict(x_test)
y_prob = model.predict_proba(x_test)[:,1]


In [ ]:
cm = confusion_matrix(y_test, y_pred)
tn, fp, fn, tp = cm.ravel()

print("True Negative:", tn)
print("False Positive:", fp)
print("False Negative:", fn)
print("True Positive:", tp)

True Negative: 4245
False Positive: 198
False Negative: 791
True Positive: 494


In [ ]:
recall = recall_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
auc = roc_auc_score(y_test, y_prob)

print(recall, precision, auc)

0.38443579766536967 0.7138728323699421 0.8344952187281878


### **Now XGBoost**

In [ ]:
import pandas as pd
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, recall_score, precision_score, roc_auc_score

In [ ]:
df = pd.read_csv("/content/credit_risk_dataset.csv")

In [ ]:
df = df.reset_index(drop=True)
df = df.dropna()
x = df.drop(columns= ['loan_status','person_home_ownership','loan_intent','loan_grade','cb_person_default_on_file'])
y = df['loan_status']
x_train, x_test, y_train, y_test = train_test_split(
    x, y, test_size=0.2, random_state=42
)

print("x_train")
display(x_train)
print(f"--------------------------------")
print("x_test")
display(x_test)
print("--------------------------------")
print("y_train")
display(y_train)
print("--------------------------------")
print("y_test")
display(y_test)
print("--------------------------------")

model = XGBClassifier(
    n_estimators=100,
    max_depth=4,
    learning_rate=0.1,
    random_state=42,
    use_label_encoder=False,
    eval_metric='logloss'
)

x_train


,person_age,person_income,person_emp_length,loan_amnt,loan_int_rate,loan_percent_income,cb_person_cred_hist_length
14324,25,105000,2.0,9600,11.71,0.09,3
5698,23,150000,3.0,5000,10.65,0.03,2
22936,34,35000,1.0,8000,6.17,0.23,9
23963,31,68000,15.0,13600,7.88,0.20,9
5750,26,25300,2.0,5400,16.02,0.21,2
...,...,...,...,...,...,...,...
24543,28,74000,12.0,5450,12.67,0.07,6
6202,23,27000,6.0,6000,11.66,0.22,3
991,22,24000,1.0,6000,12.68,0.25,4
17959,30,225000,2.0,25000,12.42,0.11,10


--------------------------------
x_test


,person_age,person_income,person_emp_length,loan_amnt,loan_int_rate,loan_percent_income,cb_person_cred_hist_length
24777,27,75400,3.0,6000,7.74,0.08,9
18341,33,53004,0.0,20000,13.04,0.38,10
14584,26,110000,3.0,4000,6.92,0.04,2
23814,29,60000,11.0,6000,15.27,0.10,10
20906,28,66725,3.0,9000,16.29,0.11,9
...,...,...,...,...,...,...,...
27436,33,140000,8.0,8000,5.79,0.06,6
9169,26,16120,2.0,5000,11.58,0.31,3
17987,34,204000,0.0,16000,11.83,0.08,5
3532,22,37000,6.0,7500,10.95,0.20,2


--------------------------------
y_train


,loan_status
14324,0
5698,0
22936,0
23963,0
5750,1
...,...
24543,0
6202,0
991,0
17959,0


--------------------------------
y_test


,loan_status
24777,0
18341,1
14584,0
23814,1
20906,1
...,...
27436,0
9169,1
17987,0
3532,0


--------------------------------


In [ ]:
model.fit(x_train, y_train)
y_pred = model.predict(x_test)
y_prob = model.predict_proba(x_test)[:,1]


/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [10:18:27] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


In [ ]:
cm = confusion_matrix(y_test, y_pred)
tn, fp, fn, tp = cm.ravel()

print("True Negative:", tn)
print("False Positive:", fp)
print("False Negative:", fn)
print("True Positive:", tp)

True Negative: 4193
False Positive: 250
False Negative: 509
True Positive: 776


In [ ]:
recall = recall_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
auc = roc_auc_score(y_test, y_prob)

print(recall, precision, auc)

0.6038910505836576 0.7563352826510721 0.8966824918487613
